# Détection & Extraction Bordereau de Prix

## Ce que fait ce notebook

À partir d'un PDF d'appel d'offres (souvent 30-100 pages), on veut trouver et extraire automatiquement le **bordereau de prix** (tableau à remplir par le soumissionnaire).

## Stratégie en 2 étapes  optimisée pour le coût

| Étape | Outil | Coût | Rôle |
|-------|-------|------|------|
| 1 | `pymupdf` | **0 €** | Scan le texte de chaque page, cherche des mots-clés typiques d'un bordereau |
| 2 | `GPT-4o vision` | **~0.002 $** | Analyse en image **uniquement la meilleure page** → extrait le tableau en JSON |

> Sans cette stratégie, envoyer tout le PDF à GPT-4o coûterait ~0.08 $ (40x plus cher).

## Structure des dossiers
```
rd/offre_financiere/
  input/
    document.pdf   ← mettre le PDF AO ici
  output/
    bordereau_extrait.json   ← résultat généré automatiquement
```

In [5]:
# Installation des dépendances si nécessaire
# pymupdf  : lecture PDF (extraction texte + rendu image)
# openai   : appel GPT-4o vision
# pillow   : manipulation d'images (utilisé en interne par pymupdf)
# !pip install pymupdf openai pillow python-dotenv pandas

In [6]:
import fitz        # pymupdf  lecture PDF
import base64       # encodage image pour l'API OpenAI
import json
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# --- Chemins ---
# INPUT  : mettre ton PDF ici avant de lancer
# OUTPUT : le JSON extrait sera sauvegardé ici automatiquement
BASE   = Path(".")
INPUT  = BASE / "input" / "CPS 27-2025.pdf"
OUTPUT = BASE / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)

# --- Clé API OpenAI ---
# Chargée depuis le fichier .env à la racine du projet (OPENAI_API_KEY=sk-...)
load_dotenv(dotenv_path=Path("../../.env"))
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("PDF trouvé :", INPUT.exists())
print("Dossier output :", OUTPUT)

PDF trouvé : True
Dossier output : output


## Étape 1  Détection par mots-clés (gratuit, pymupdf)

**Principe :** on lit le texte brut de chaque page et on compte combien de mots-clés caractéristiques d'un bordereau elle contient.

- Score ≥ 2 → page candidate
- La page avec le score le plus élevé sera envoyée à GPT-4o

**Pourquoi un seuil de 2 ?** Pour éviter les faux positifs  un tableau de synthèse peut contenir "prix total" sans être le bordereau officiel.

In [7]:
# Mots-clés caractéristiques d'un bordereau de prix dans les AO marocains
# On met en minuscules pour une comparaison insensible à la casse
KEYWORDS = [
    "bordereau",
    "prix unitaire",
    "hors tva",
    "désignation",
    "désignations",
    "montant total",
    "unité de compte",
    "prix-detail",
    "détail estimatif",
    "forfait",
    "prix total",
]

def detecter_pages_bordereau(pdf_path: Path, seuil: int = 2) -> list[dict]:
    """
    INPUT  : chemin vers le PDF
    OUTPUT : liste de dicts { page_index, page_num, score, mots_trouves }
             triée par score décroissant, filtrée par seuil minimum
    """
    doc = fitz.open(str(pdf_path))
    pages_detectees = []

    for i, page in enumerate(doc):
        # get_text() extrait le texte embarqué dans le PDF (pas d'OCR ici)
        texte = page.get_text().lower()
        mots_trouves = [kw for kw in KEYWORDS if kw in texte]
        score = len(mots_trouves)

        print(f"Page {i+1:>3} | score={score} | {mots_trouves}")

        if score >= seuil:
            pages_detectees.append({
                "page_index": i,        # index 0-based pour pymupdf
                "page_num": i + 1,      # numéro lisible pour l'humain
                "score": score,
                "mots_trouves": mots_trouves,
            })

    doc.close()
    # Trier par score décroissant pour avoir la meilleure en premier
    pages_detectees.sort(key=lambda p: p["score"], reverse=True)
    return pages_detectees


# --- Lancement ---
pages = detecter_pages_bordereau(INPUT)

print(f"\n{'='*50}")
print(f"→ {len(pages)} page(s) candidate(s) trouvée(s) :")
for p in pages:
    print(f"   Page {p['page_num']:>3} | score={p['score']} | {p['mots_trouves']}")

Page   1 | score=0 | []
Page   2 | score=2 | ['bordereau', 'prix-detail']
Page   3 | score=0 | []
Page   4 | score=2 | ['bordereau', 'détail estimatif']
Page   5 | score=0 | []
Page   6 | score=0 | []
Page   7 | score=0 | []
Page   8 | score=2 | ['bordereau', 'prix unitaire']
Page   9 | score=2 | ['bordereau', 'montant total']
Page  10 | score=0 | []
Page  11 | score=0 | []
Page  12 | score=0 | []
Page  13 | score=0 | []
Page  14 | score=0 | []
Page  15 | score=0 | []
Page  16 | score=0 | []
Page  17 | score=8 | ['bordereau', 'prix unitaire', 'hors tva', 'désignation', 'désignations', 'montant total', 'forfait', 'prix total']
Page  18 | score=0 | []

→ 5 page(s) candidate(s) trouvée(s) :
   Page  17 | score=8 | ['bordereau', 'prix unitaire', 'hors tva', 'désignation', 'désignations', 'montant total', 'forfait', 'prix total']
   Page   2 | score=2 | ['bordereau', 'prix-detail']
   Page   4 | score=2 | ['bordereau', 'détail estimatif']
   Page   8 | score=2 | ['bordereau', 'prix unitaire

## Étape 2  Extraction GPT-4o vision

**Principe :**
1. On convertit la page PDF en image PNG haute résolution (zoom x2)
2. On envoie cette image à GPT-4o avec un prompt qui lui demande d'extraire le tableau tel quel
3. GPT-4o retourne un JSON structuré  les cellules vides (à remplir par le soumissionnaire) ont la valeur `null`

**Coût réel :** affiché après chaque appel (~0.002 $ par page)

In [8]:
def page_to_base64(pdf_path: Path, page_index: int, zoom: float = 2.0) -> str:
    """
    Convertit une page PDF en image PNG encodée en base64.

    INPUT  : chemin PDF, numéro de page (0-based), zoom (2.0 = 144 DPI  bon compromis qualité/taille)
    OUTPUT : string base64 prête à envoyer à l'API OpenAI
    """
    doc = fitz.open(str(pdf_path))
    page = doc[page_index]
    mat = fitz.Matrix(zoom, zoom)   # zoom x2 → image plus nette pour GPT-4o
    pix = page.get_pixmap(matrix=mat)
    img_bytes = pix.tobytes("png")
    doc.close()
    return base64.b64encode(img_bytes).decode("utf-8")


# --- Prompt système : contexte métier ---
SYSTEM_PROMPT = """Tu es un assistant spécialisé dans l'analyse de documents d'appels d'offres marocains.
Tu extrais avec précision les tableaux de bordereaux de prix.
Réponds uniquement en JSON valide, sans markdown."""

# --- Prompt utilisateur : instruction d'extraction ---
# On demande à GPT-4o de s'adapter aux colonnes réelles du document
# sans imposer une structure fixe (car chaque AO a son propre format)
USER_PROMPT = """Cette page contient un tableau de bordereau de prix issu d'un appel d'offres.

Extrait le tableau tel qu'il apparaît dans le document, en t'adaptant aux colonnes réellement présentes.

Retourne un JSON avec cette structure :
{
  "titre": "titre exact du tableau tel qu'il apparaît dans le document",
  "colonnes": ["liste", "des", "colonnes", "telles", "quelles"],
  "lignes": [
    {
      "valeurs": ["valeur col1", "valeur col2", "..."]
    }
  ],
  "totaux": [
    {"label": "Montant Total Hors TVA", "valeur": null},
    {"label": "Taux de la TVA (20%)", "valeur": "20%"},
    {"label": "Montant Total TTC", "valeur": null}
  ]
}

Règles :
- Respecte l'ordre et le nombre exact de colonnes du document
- Une ligne = une ligne du tableau (y compris les lignes de sous-total si présentes)
- Les cellules vides (à remplir par le soumissionnaire) ont la valeur null
- Les cellules renseignées dans le document gardent leur valeur exacte (texte, chiffre, "FORFAIT", etc.)
- Ne déduis rien, ne complète rien  recopie uniquement ce qui est visible"""


def extraire_bordereau_gpt4o(pdf_path: Path, page_index: int) -> dict:
    """
    INPUT  : chemin PDF + numéro de page (0-based)
    OUTPUT : dict JSON avec { titre, colonnes, lignes, totaux }
             + affiche le coût réel de l'appel API
    """
    img_b64 = page_to_base64(pdf_path, page_index)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": USER_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{img_b64}",
                            "detail": "high"   # high = GPT-4o analyse l'image en haute résolution
                        }
                    }
                ]
            }
        ],
        max_tokens=2000,
        temperature=0,   # 0 = réponse déterministe, pas de créativité
    )

    raw = response.choices[0].message.content.strip()
    # GPT-4o met parfois du markdown (```json ... ```) malgré le system prompt → on le nettoie
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]

    # Affichage du coût réel
    usage = response.usage
    cout = usage.prompt_tokens * 0.0000025 + usage.completion_tokens * 0.00001
    print(f"  Tokens : {usage.prompt_tokens} prompt + {usage.completion_tokens} completion")
    print(f"  Coût   : ~${cout:.4f}")

    return json.loads(raw)

In [9]:
# --- Lancement : on prend uniquement la page avec le score le plus élevé ---
# Raison : envoyer plusieurs pages à GPT-4o multiplierait le coût inutilement.
# La page avec le plus de mots-clés est quasi-certainement le bon bordereau.

resultats = []

if not pages:
    print("Aucune page bordereau détectée. Vérifie le PDF ou baisse le seuil dans detecter_pages_bordereau().")
else:
    best = pages[0]  # déjà triées par score décroissant à l'étape 1
    print(f"Page retenue  : page {best['page_num']} (score={best['score']})")
    print(f"Mots trouvés  : {best['mots_trouves']}")
    print(f"\nEnvoi à GPT-4o vision...")

    try:
        bordereau = extraire_bordereau_gpt4o(INPUT, best["page_index"])
        bordereau["_page"] = best["page_num"]   # on garde le numéro de page pour référence
        resultats.append(bordereau)
        print(f"\n✓ {len(bordereau['lignes'])} ligne(s) extraite(s)")
    except Exception as e:
        print(f"✗ Erreur extraction : {e}")

# --- Sauvegarde ---
# OUTPUT : rd/offre_financiere/output/bordereau_extrait.json
output_file = OUTPUT / "bordereau_extrait.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2)

print(f"\nJSON sauvegardé : {output_file}")

Page retenue  : page 17 (score=8)
Mots trouvés  : ['bordereau', 'prix unitaire', 'hors tva', 'désignation', 'désignations', 'montant total', 'forfait', 'prix total']

Envoi à GPT-4o vision...
  Tokens : 1446 prompt + 352 completion
  Coût   : ~$0.0071

✓ 5 ligne(s) extraite(s)

JSON sauvegardé : output\bordereau_extrait.json


## Résultat  Affichage du tableau extrait

Le tableau est affiché sous forme de DataFrame pandas.
- Les cellules `None` = vides dans le document original (à remplir par le soumissionnaire)
- Les totaux (HT, TVA, TTC) sont affichés séparément en bas

In [10]:
import pandas as pd

for b in resultats:
    print(f"Page {b['_page']}  {b.get('titre', 'Bordereau')}")
    print()

    # Reconstruction du DataFrame depuis { colonnes, lignes: [{valeurs: [...]}] }
    colonnes = b.get("colonnes", [])
    lignes   = [l["valeurs"] for l in b.get("lignes", [])]
    df = pd.DataFrame(lignes, columns=colonnes if colonnes else None)

    # Affichage avec None visible (pas remplacé par NaN pour rester lisible)
    display(df)

    # Affichage des totaux (HT, TVA, TTC)
    if b.get("totaux"):
        print()
        for t in b["totaux"]:
            valeur = t["valeur"] if t["valeur"] is not None else "[ à remplir ]"
            print(f"  {t['label']:<35} {valeur}")

    # Résumé : combien de cellules restent à remplir
    nulls = sum(1 for l in b.get("lignes", []) for v in l["valeurs"] if v is None)
    print(f"\n→ {nulls} cellule(s) à remplir par le soumissionnaire")

Page 17  ARTICLE 34. BORDEREAU DU PRIX-DETAIL ESTIMATIF



,N° du prix,Désignations,Unité de compte,Quantité (1),Prix unitaire en DH hors TVA (2),Prix total Hors TVA (en chiffres) 3=1x2
0,1,Etape 1 : Atelier de démarrage,FORFAIT,1,None,None
1,2,Etape 2 : diagnostic et première visite des pé...,FORFAIT,1,None,None
2,3,Etape 3 : Deuxième visite des pépinières,FORFAIT,1,None,None
3,4,Etape 4 : Ateliers de mise à niveau des pépini...,FORFAIT,1,None,None
4,5,Etape 5 : Atelier de clôture,FORFAIT,1,None,None



  Montant Total Hors TVA              [ à remplir ]
  Taux de la TVA (20%)                20%
  Montant Total TTC                   [ à remplir ]

→ 10 cellule(s) à remplir par le soumissionnaire


## Export Excel

Génère un fichier `.xlsx` avec le tableau extrait.
Les cellules à remplir (valeur `null`) apparaissent en **jaune** pour guider le soumissionnaire.

In [12]:
# !pip install openpyxl
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

def exporter_excel(bordereau: dict, output_path: Path) -> None:
    """
    INPUT  : dict bordereau (issu de GPT-4o) + chemin de sortie .xlsx
    OUTPUT : fichier Excel avec :
               - titre en gras en haut
               - en-têtes de colonnes surlignés en bleu
               - cellules vides (null) surlignées en jaune = à remplir
               - ligne de totaux en bas
    """
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Bordereau de prix"

    # --- Styles ---
    style_titre   = Font(bold=True, size=13)
    style_header  = Font(bold=True, color="FFFFFF")
    fill_header   = PatternFill("solid", fgColor="2E4057")   # bleu foncé
    fill_vide     = PatternFill("solid", fgColor="FFF176")   # jaune = à remplir
    fill_total    = PatternFill("solid", fgColor="E8F5E9")   # vert clair = totaux
    border_thin   = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"),  bottom=Side(style="thin")
    )
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left   = Alignment(horizontal="left",   vertical="center", wrap_text=True)

    row = 1

    # --- Titre ---
    titre = bordereau.get("titre", "Bordereau de prix")
    ws.cell(row=row, column=1, value=titre).font = style_titre
    row += 2  # ligne vide après le titre

    # --- En-têtes de colonnes ---
    colonnes = bordereau.get("colonnes", [])
    for col_idx, col_name in enumerate(colonnes, start=1):
        cell = ws.cell(row=row, column=col_idx, value=col_name)
        cell.font      = style_header
        cell.fill      = fill_header
        cell.alignment = center
        cell.border    = border_thin
    row += 1

    # --- Lignes du tableau ---
    for ligne in bordereau.get("lignes", []):
        valeurs = ligne.get("valeurs", [])
        for col_idx, valeur in enumerate(valeurs, start=1):
            cell = ws.cell(row=row, column=col_idx, value=valeur)
            cell.border    = border_thin
            cell.alignment = center if col_idx != 2 else left  # col 2 = désignation → alignée gauche
            if valeur is None:
                cell.fill = fill_vide   # jaune = cellule vide à remplir
        row += 1

    # --- Totaux ---
    totaux = bordereau.get("totaux", [])
    if totaux:
        row += 1  # ligne vide de séparation
        for total in totaux:
            # Label dans l'avant-dernière colonne, valeur dans la dernière
            nb_cols = len(colonnes)
            label_cell = ws.cell(row=row, column=nb_cols - 1, value=total["label"])
            label_cell.font      = Font(bold=True)
            label_cell.fill      = fill_total
            label_cell.border    = border_thin
            label_cell.alignment = right_align = Alignment(horizontal="right")

            val_cell = ws.cell(row=row, column=nb_cols, value=total["valeur"])
            val_cell.fill   = fill_vide if total["valeur"] is None else fill_total
            val_cell.border = border_thin
            val_cell.alignment = center
            row += 1

    # --- Largeur automatique des colonnes ---
    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = min(max_len + 4, 50)

    wb.save(str(output_path))
    print(f"✓ Excel sauvegardé : {output_path}")


# --- Lancement ---
if resultats:
    b = resultats[0]
    excel_path = OUTPUT / f"bordereau_page{b['_page']}.xlsx"
    exporter_excel(b, excel_path)
else:
    print("Aucun résultat à exporter.")

✓ Excel sauvegardé : output\bordereau_page17.xlsx
